# Large toy model experiments

We now use a larger toy model with correlated features to validate that our hedging metric peforms as expected. We'll use a toy model with 50 true features with random correlations.

In [ ]:
from functools import partial
from pathlib import Path
import torch
import pandas as pd
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt

from hedging_paper.toy_models.get_training_batch import generate_random_correlation_matrix, get_training_batch
from hedging_paper.toy_models.toy_model import ToyModel
from hedging_paper.toy_models.plotting import SEABORN_RC_CONTEXT
from hedging_paper.util import DEFAULT_DEVICE

tqdm._instances.clear()  # type: ignore

feat_probs = 0.345 * (50 - torch.arange(50) - 1) / 50 + 0.05

if Path("correlations.pt").exists():
    print("Loading correlations from disk")
    correlations = torch.load("correlations.pt")
else:
    correlations = generate_random_correlation_matrix(
        correlation_strength_range=(0.3, 0.9),
        num_features=50,
        seed=42,
    )
    torch.save(correlations, "correlations.pt")

indices = torch.arange(50) + 1
df = pd.DataFrame({
    "P_i": feat_probs,
    "feature": map(str, indices.tolist()),
})


if Path("toy_model.pt").exists():
    print("Loading toy model from disk")
    toy_model = torch.load("toy_model.pt", weights_only=False)
else:
    toy_model = ToyModel(num_feats=50, hidden_dim=100).to(DEFAULT_DEVICE)
    torch.save(toy_model, "toy_model.pt")


generate_batch = partial(
    get_training_batch,
    firing_probabilities=feat_probs,
    std_firing_magnitudes=torch.ones_like(feat_probs) * 0.15,
    correlation_matrix=correlations,
)

Path("plots/toy_setup").mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 150})
with plt.rc_context(SEABORN_RC_CONTEXT):
    plt.figure(figsize=(3, 2))
    sns.barplot(data=df, x="feature", y="P_i")
    plt.xlabel("Feature")
    plt.ylabel("$P_i$")
    plt.title("Feature firing probabilities $P_i$")
    # Increase tick spacing to prevent overlapping
    plt.xticks(range(0, len(df), 5), [str(i) for i in range(0, len(df), 5)])  # Show every 5th tick, 0-indexed
    plt.tight_layout()
    plt.savefig("plots/toy_setup/toy_model_feature_firing_probabilities.pdf")
    plt.show()

plt.rcParams.update({"figure.dpi": 150})
with plt.rc_context(SEABORN_RC_CONTEXT):
    plt.figure(figsize=(2.5, 2))
    sns.heatmap(correlations, cmap="RdBu", center=0, vmin=-1, vmax=1)
    plt.xlabel("Feature")
    plt.ylabel("Feature")
    plt.title("Feature correlation matrix")
    # Increase tick spacing to prevent overlapping
    plt.xticks(range(0, len(correlations), 10), [str(i) for i in range(0, len(correlations), 10)])  # Show every 10th tick, 0-indexed
    plt.yticks(range(0, len(correlations), 10), [str(i) for i in range(0, len(correlations), 10)])  # Show every 10th tick, 0-indexed
    # plt.tight_layout()
    plt.savefig("plots/toy_setup/toy_model_correlation_matrix.pdf")
    plt.show()

## Finding the True L0

Next, we calculate the true L0 for this dataset (spoiler: it's ~11)

In [ ]:
sample = generate_batch(100_000)
true_l0 = (sample > 0).float().sum(dim=-1).mean()
print(f"True L0: {true_l0}")

Next, we'll train a sweep toy SAEs at different widths and run our hedging metric on them.

In [ ]:
from collections import defaultdict
from hedging_paper.saes.batch_topk_sae import BatchTopkSAE
from hedging_paper.toy_models.train_toy_sae import train_toy_sae
from hedging_paper.saes.base_sae import BaseSAERunnerConfig
from hedging_paper.saes.base_sae import BaseSAEConfig
from hedging_paper.train_sae import load_pretrained_weights

EXTENSION_N = 2

btk_saes_by_width = defaultdict(lambda: defaultdict(list))
for seed in [0, 1, 2, 3]:
    for width in range(15, 51, 5):
        initial_sae_path = f"btk_saes_by_width/seed_{seed}/{width}/initial"
        control_sae_path = f"btk_saes_by_width/seed_{seed}/{width}/control"
        extended_sae_path = f"btk_saes_by_width/seed_{seed}/{width}/extended-{EXTENSION_N}"

        cfg = BaseSAERunnerConfig(
            architecture="topk",
            activation_fn_kwargs={"k": true_l0.item()},  # type: ignore
            context_size=500,
            d_in=toy_model.embed.weight.shape[0],
            d_sae=width,
            normalize_sae_decoder=False,
            scale_sparsity_penalty_by_decoder_norm=True,
            init_encoder_as_decoder_transpose=True,
            apply_b_dec_to_input=True,
            b_dec_init_method="zeros",
            extend_sae_latents=0,
            extend_sae_path=None,
        )
        
        # Train initial SAE
        if Path(initial_sae_path).exists():
            print(f"Loading initial SAE with width={width}, seed={seed} from disk")
            initial_sae = BatchTopkSAE.load_from_disk(initial_sae_path)
        else:
            print(f"Training initial SAE with width={width}, seed={seed}")
            initial_sae = BatchTopkSAE(BaseSAEConfig.from_sae_runner_config(cfg))
            train_toy_sae(initial_sae, toy_model, generate_batch)
            Path(initial_sae_path).mkdir(parents=True, exist_ok=True)
            initial_sae.save_model(initial_sae_path)
        btk_saes_by_width[width]["initial"].append(initial_sae)
        
        # Train control SAE (continue training from initial, no extension)
        if Path(control_sae_path).exists():
            print(f"Loading control SAE with width={width}, seed={seed} from disk")
            control_sae = BatchTopkSAE.load_from_disk(control_sae_path)
        else:
            print(f"Training control SAE with width={width}, seed={seed}")
            control_sae = BatchTopkSAE(BaseSAEConfig.from_sae_runner_config(cfg))
            load_pretrained_weights(control_sae, initial_sae_path, "cpu")
            train_toy_sae(control_sae, toy_model, generate_batch)
            Path(control_sae_path).mkdir(parents=True, exist_ok=True)
            control_sae.save_model(control_sae_path)
        btk_saes_by_width[width]["control"].append(control_sae)
        
        # Train extended SAE (extend initial by EXTENSION_N latents)
        if Path(extended_sae_path).exists():
            print(f"Loading extended SAE with width={width}, seed={seed} from disk")
            extended_sae = BatchTopkSAE.load_from_disk(extended_sae_path)
        else:
            print(f"Training extended SAE with width={width}, seed={seed}")
            extended_sae = BatchTopkSAE(BaseSAEConfig.from_sae_runner_config(cfg))
            load_pretrained_weights(extended_sae, initial_sae_path, "cpu")
            extended_sae.extend_sae(EXTENSION_N)
            train_toy_sae(extended_sae, toy_model, generate_batch)
            Path(extended_sae_path).mkdir(parents=True, exist_ok=True)
            extended_sae.save_model(extended_sae_path)
        btk_saes_by_width[width]["extended"].append(extended_sae)

In [ ]:
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns

from hedging_paper.evals.hedging_eval import calculate_hedging_stats

N_BASELINES = 50

data = []

for width, saes in btk_saes_by_width.items():
    control_saes = saes["control"]
    extended_saes = saes["extended"]
    for seed, (control_sae, extended_sae) in enumerate(zip(control_saes, extended_saes)):
        data.append({
            "seed": seed,
            **calculate_hedging_stats(control_sae, extended_sae, N_BASELINES),
        })

df = pd.DataFrame(data)

px.line(
    df,
    x="width",
    y="hedging_rate",
    color="seed",
    hover_data=[
        "width",
        "nlatents",
        "delta_norm",
        "delta_proj",
        "delta_proj_portion",
        "control_dec_norm",
        "mean_dec_cos_sim_delta",
        "nbaselines",
        "baseline_proj",
        "baseline_proj_std",
        "baseline_proj_portion",
    ],
).show()

plt.rcParams.update({"figure.dpi": 150})
sns.set_theme()
with plt.rc_context(SEABORN_RC_CONTEXT):
    plt.figure(figsize=(3, 1.5))
    sns.lineplot(data=df, x="width", y="hedging_rate")
    plt.grid(True, alpha=0.3)    
    plt.xlabel("SAE width")
    plt.ylabel("Hedging degree")
    plt.title("Hedging degree vs SAE width")
    plt.tight_layout()
    plt.savefig("plots/btk_hedging_vs_width.pdf")
    plt.show()